# Wisconsin Card Sort Test v3

**Track:** Executive Functions
**Cognitive Ability:** Set-shifting / cognitive flexibility

## Methodology

Tests the ability to discover sorting rules from feedback and flexibly shift when rules change.

### v3 Changes (breaking ceiling effects):
- **5 sorting dimensions** (color, shape, number, border style, background shade)
- **Hidden dimensions:** Model is NOT told which dimensions exist — must discover from feedback
- **Probabilistic feedback:** 85% reliable (15% chance of random feedback)
- **Variable shift criterion:** Rule shifts after 3-7 consecutive correct (not fixed)
- **Multi-dimensional phase:** Later blocks require matching on 2 dimensions simultaneously

### Scoring
- 0.30 × single-dimension accuracy + 0.45 × multi-dimension accuracy + 0.25 × perseveration resistance

### References
- Grant & Berg (1948), Milner (1963), Barceló (2003)


In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null

In [ ]:
"""
WCST v3 — Hidden dimensions, probabilistic feedback, variable shifts.

Changes from v2:
- 5 dimensions (added border_style, background): model must discover which matter
- Model is NOT told what dimensions exist — must infer from feedback
- Probabilistic feedback: 85% reliable (15% random)
- Variable shift criterion: 3-7 consecutive correct before shift
- Multi-dimensional phase: later blocks require matching on 2 dimensions
- 80 total cards across phases
"""

import random
import hashlib

COLORS = ["red", "blue", "green", "yellow"]
SHAPES = ["circle", "triangle", "square", "star"]
NUMBERS = [1, 2, 3, 4]
BORDERS = ["solid", "dashed", "dotted"]
BACKGROUNDS = ["light", "dark", "striped"]

ALL_DIMS = ["color", "shape", "number", "border", "background"]

# Reference cards — each unique on all 5 dimensions
REFERENCE_CARDS = [
    {"color": "red",    "shape": "circle",   "number": 1, "border": "solid",  "background": "light"},
    {"color": "blue",   "shape": "triangle", "number": 2, "border": "dashed", "background": "dark"},
    {"color": "green",  "shape": "square",   "number": 3, "border": "dotted", "background": "striped"},
    {"color": "yellow", "shape": "star",     "number": 4, "border": "solid",  "background": "dark"},
]


def card_str(card):
    n = card["number"]
    return (f"{n} {card['color']} {card['shape']}{'s' if n > 1 else ''}, "
            f"{card['border']} border, {card['background']} background")


def _match_ref(target, rule_dims):
    """Return 1-indexed reference card matching target on rule dimensions."""
    for i, ref in enumerate(REFERENCE_CARDS):
        if all(target[d] == ref[d] for d in rule_dims):
            return i + 1
    return None


def _make_target(rng, rule_dims, correct_ref_idx):
    """Generate target matching ref[correct_ref_idx] on rule_dims, different on others."""
    ref = REFERENCE_CARDS[correct_ref_idx]
    other_indices = [i for i in range(4) if i != correct_ref_idx]
    rng.shuffle(other_indices)
    
    card = {}
    # Match on rule dimensions
    for d in rule_dims:
        card[d] = ref[d]
    
    # Differ on other dimensions
    other_dims = [d for d in ALL_DIMS if d not in rule_dims]
    for j, d in enumerate(other_dims):
        other_ref = REFERENCE_CARDS[other_indices[j % len(other_indices)]]
        card[d] = other_ref[d]
    
    return card


def generate_wcst_v3(seed="wcst_v3_seed"):
    """Generate WCST v3 trial sequence."""
    rng = random.Random(int(hashlib.sha256(seed.encode()).hexdigest(), 16))
    
    phases = []
    
    # Phase 1: Single-dimension sorting (5 rule shifts)
    # Rules cycle through dimensions, model must discover which one
    single_rules = [["color"], ["shape"], ["number"], ["border"], ["background"]]
    rng.shuffle(single_rules)
    
    for rule_dims in single_rules[:5]:
        shift_after = rng.randint(3, 7)  # Variable shift criterion
        
        # Generate history (correct examples under this rule)
        history = []
        for _ in range(rng.randint(3, 5)):
            ref_idx = rng.randint(0, 3)
            target = _make_target(rng, rule_dims, ref_idx)
            correct = ref_idx + 1
            # Probabilistic feedback
            if rng.random() < 0.85:
                feedback = "Correct"
            else:
                feedback = "Incorrect"  # Noisy feedback
            history.append({
                "target": target,
                "chosen": correct,
                "feedback": feedback,
                "actual_correct": correct,
            })
        
        # Generate test cards
        n_test = rng.randint(6, 10)
        test_cards = []
        for _ in range(n_test):
            ref_idx = rng.randint(0, 3)
            target = _make_target(rng, rule_dims, ref_idx)
            test_cards.append({
                "target": target,
                "correct": ref_idx + 1,
            })
        
        phases.append({
            "phase_type": "single",
            "rule_dims": rule_dims,
            "history": history,
            "test_cards": test_cards,
            "shift_criterion": shift_after,
        })
    
    # Phase 2: Multi-dimensional sorting (2 blocks, require matching 2 dims)
    multi_rules = [["color", "number"], ["shape", "background"]]
    for rule_dims in multi_rules:
        history = []
        for _ in range(rng.randint(4, 6)):
            ref_idx = rng.randint(0, 3)
            target = _make_target(rng, rule_dims, ref_idx)
            correct = ref_idx + 1
            feedback = "Correct" if rng.random() < 0.85 else "Incorrect"
            history.append({
                "target": target,
                "chosen": correct,
                "feedback": feedback,
                "actual_correct": correct,
            })
        
        test_cards = []
        for _ in range(rng.randint(6, 8)):
            ref_idx = rng.randint(0, 3)
            target = _make_target(rng, rule_dims, ref_idx)
            test_cards.append({"target": target, "correct": ref_idx + 1})
        
        phases.append({
            "phase_type": "multi",
            "rule_dims": rule_dims,
            "history": history,
            "test_cards": test_cards,
        })
    
    return {"phases": phases, "reference_cards": REFERENCE_CARDS}


WCST_V3 = generate_wcst_v3()

In [ ]:
"""
WCST v3 — Hidden dimensions, probabilistic feedback, variable shifts.

Changes from v2:
- 5 sorting dimensions (color, shape, number, border, background)
- Model NOT told which dimensions exist — must discover from feedback
- Probabilistic feedback (85% reliable)
- Variable shift criterion (3-7 correct before shift)
- Multi-dimensional phase (match on 2 dims simultaneously)

Cognitive Basis:
- Grant & Berg (1948): Original WCST
- Milner (1963): Perseveration in frontal lobe patients
- Barceló (2003): Probabilistic feedback variants
"""

import kaggle_benchmarks as kbench
import re
import json as _json
import numpy as np


def _safe_log(data): print(_json.dumps(data, indent=2, default=str))


def _strip_think(text: str) -> str:
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def _parse_responses(raw: str, n_expected: int) -> list:
    """Parse card number responses (1-4)."""
    raw = _strip_think(raw)
    raw = re.sub(r'//.*', '', raw)
    
    # Try numbered lines
    results = []
    for line in raw.strip().split('\n'):
        line = line.strip()
        m = re.search(r'\b([1-4])\b', line)
        if m:
            results.append(int(m.group(1)))
            if len(results) >= n_expected:
                break
    
    if len(results) >= n_expected:
        return results[:n_expected]
    
    # Fallback: all numbers 1-4 in order
    all_nums = re.findall(r'\b([1-4])\b', raw)
    # Take last n_expected (more likely to be final answers)
    nums = [int(x) for x in all_nums]
    if len(nums) >= n_expected:
        return nums[-n_expected:]
    
    # Pad with 1
    while len(nums) < n_expected:
        nums.append(1)
    return nums[:n_expected]


def run_phase(llm, phase: dict, phase_idx: int) -> dict:
    """Run one WCST phase (single or multi-dimensional sorting)."""
    history = phase["history"]
    test_cards = phase["test_cards"]
    rule_dims = phase["rule_dims"]
    
    # Build reference card display (show ALL 5 dimensions, don't reveal which matter)
    ref_display = "\n".join(
        f"  Card {i+1}: {card_str(ref)}"
        for i, ref in enumerate(REFERENCE_CARDS)
    )
    
    # Build history display with feedback
    history_lines = []
    for h in history:
        history_lines.append(
            f"  Target: {card_str(h['target'])} → Sorted to Card {h['chosen']} → {h['feedback']}"
        )
    
    # Build test card display
    test_lines = []
    for i, tc in enumerate(test_cards):
        test_lines.append(f"  {i+1}. {card_str(tc['target'])}")
    
    phase_label = "Multi-Dimension" if phase["phase_type"] == "multi" else "Single-Dimension"
    
    prompt = (
        f"CARD SORTING TASK — Phase {phase_idx + 1} ({phase_label})\n\n"
        f"You have 4 reference cards:\n{ref_display}\n\n"
        f"Each card has multiple properties. The sorting rule uses one or more properties "
        f"to determine which reference card a target matches.\n\n"
        f"Here is the recent sorting history with feedback:\n"
        + "\n".join(history_lines) + "\n\n"
        f"Based on the pattern in the feedback, sort each new target card.\n"
        f"For each card, respond with the reference card number (1-4).\n\n"
        f"New cards to sort:\n" + "\n".join(test_lines) + "\n\n"
        f"Respond with {len(test_cards)} numbers (1-4), one per line."
    )
    
    with kbench.chats.new(f"wcst_phase_{phase_idx}"):
        raw = llm.prompt(prompt)
    
    choices = _parse_responses(raw, len(test_cards))
    
    # Score
    correct_count = 0
    results = []
    for i, (tc, choice) in enumerate(zip(test_cards, choices)):
        is_correct = (choice == tc["correct"])
        if is_correct:
            correct_count += 1
        results.append({
            "card": i + 1,
            "correct_ref": tc["correct"],
            "model_choice": choice,
            "correct": is_correct,
        })
    
    accuracy = correct_count / max(len(test_cards), 1)
    
    return {
        "phase_type": phase["phase_type"],
        "rule_dims": rule_dims,
        "accuracy": round(accuracy, 4),
        "n_correct": correct_count,
        "n_test": len(test_cards),
        "results": results,
    }


@kbench.task(name="Wisconsin Card Sort")
def exec_func_wcst(llm) -> float:
    """
    WCST v3 — Hidden dimensions, probabilistic feedback, variable shifts.
    
    Score = 0.30 * single_phase_mean + 0.45 * multi_phase_mean + 0.25 * perseveration_resistance
    """
    data = WCST_V3
    phase_results = []
    
    for i, phase in enumerate(data["phases"]):
        result = run_phase(llm, phase, i)
        phase_results.append(result)
    
    # Separate single and multi-dim phases
    single_accs = [r["accuracy"] for r in phase_results if r["phase_type"] == "single"]
    multi_accs = [r["accuracy"] for r in phase_results if r["phase_type"] == "multi"]
    
    single_mean = sum(single_accs) / max(len(single_accs), 1)
    multi_mean = sum(multi_accs) / max(len(multi_accs), 1)
    
    # Perseveration resistance: accuracy should not drop across phases
    if len(single_accs) >= 2:
        first_half = single_accs[:len(single_accs)//2]
        second_half = single_accs[len(single_accs)//2:]
        first_mean = sum(first_half) / len(first_half)
        second_mean = sum(second_half) / len(second_half)
        # If second half is worse, perseveration penalty
        persev_resistance = min(1.0, second_mean / max(first_mean, 0.01))
    else:
        persev_resistance = single_mean
    
    score = 0.30 * single_mean + 0.45 * multi_mean + 0.25 * persev_resistance
    score = round(float(np.clip(score, 0, 1)), 4)
    
    print(f"\n{'='*60}")
    print(f"WCST v3 RESULTS")
    print(f"{'='*60}")
    for r in phase_results:
        dims = "+".join(r["rule_dims"])
        print(f"  {r['phase_type']} [{dims}]: {r['accuracy']:.2%} ({r['n_correct']}/{r['n_test']})")
    print(f"\n  Single-dim mean: {single_mean:.2%}")
    print(f"  Multi-dim mean:  {multi_mean:.2%}")
    print(f"  Perseveration resistance: {persev_resistance:.2%}")
    print(f"  Composite score: {score:.4f}")
    
    _safe_log({
        "benchmark": "WCST v3",
        "phases": [{
            "type": r["phase_type"],
            "dims": r["rule_dims"],
            "accuracy": r["accuracy"],
        } for r in phase_results],
        "composite_score": score,
    })
    
    return score

In [ ]:
exec_func_wcst.run(llm=kbench.llm)